# CE / 标签平滑比较与学生后校准

2026-10-23 · CLIP–Jev · 中文分步 Colab notebook · 代码全部内嵌

**目标 1：** 区分标签平滑的一般正则化作用与 Jev 蒸馏收益。三组 CE、LS（ε=0.1）、Jev raw KD（α=0.75，τ=2）均重新训练三个种子，共 **9 次训练**。CE 是本轮同环境重训的对照，不把历史 CE 当作严格匹配的新基线。

**目标 2：** 对每个选定学生单独做温度后校准，比较 raw 与 temperature 的 NLL、Brier、ECE。它作用于**学生输出**，与“先校准 Teacher 再蒸馏”的 KD_calibrated 不同。正温度不改变 argmax，因此不应提高 Accuracy/Macro-F1。

默认只出开发阶段验证报告；历史测试复评单独开关且默认关闭。不新增或使用 Winoground。本 notebook 尚未在 Colab 上执行正式训练，运行后才会得到新成绩。

### 输入准备

1. 将 `1013实验成果包与结论.zip` 解压到 Drive，保留 `文件校验清单.json`、`reused_v1` 等目录。
2. 准备完整 Flickr30k 原图目录：原成果包的 24 张案例图片不够。需要 val_fit 的 400 张图及 val_select 前 32 张兼容检查图；若做历史测试，还需 1000 张测试图。
3. Colab 选择 GPU。沿用当前 PyTorch；所有新组统一 FP32、禁用 TF32，环境会锁定。不是对历史 torch/GPU 的逐位复现。
4. 修改第 2 节的路径。输入只读，新特征/断点/报告写入新 WORK。


## 1．安装特征提取依赖

保留当前 PyTorch/numpy，仅固定与历史预处理一致的 Transformers/Pillow/Hugging Face Hub。首次若安装依赖，会提示重启会话；重启后重新运行本格。无需下载或运行 Teacher。


In [ ]:
import os, sys, subprocess, importlib.metadata as md
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
pins={'transformers':'4.57.1','huggingface_hub':'0.36.0','Pillow':'11.3.0'}
def version(name):
    try: return md.version(name)
    except md.PackageNotFoundError: return None
install=[f'{name}=={v}' for name,v in pins.items() if version(name)!=v]
if version('pandas') is None: install.append('pandas')
if install:
    subprocess.run([sys.executable,'-m','pip','install',*install],check=True)
    raise RuntimeError('依赖已安装。请重启会话，再从本格继续。')
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), '请在 Colab 选择 GPU 运行时。'
print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'numpy:',np.__version__)


## 2．路径与本轮固定方案

`SOURCE` 指向已解压的 10月13日成果包或其中的 reused_v1；`IMAGES` 下直接存放清单中的 jpg 文件。`WORK` 要与原包相互独立。

默认 ε=0.1 是预先固定的单个标签平滑强度，不扫描 ε，不把它称为最优值。PyTorch 的定义是 `(1−ε)·one_hot + ε/3`，真类目标为 0.9333，其余为 0.0333。

固定 30 epochs、batch=128、AdamW lr=1e-3、weight_decay=1e-4；冻结 CLIP，只训练 2048→512→3 的 MLP（ReLU，dropout=0.1）。按 val_select 未校准 Macro-F1 最大、NLL 最小、较早 epoch 选模。三个方法的初始化及每轮批次顺序按种子配对。

断线后保持同一 WORK 和配置，从头依次重跑；每轮保存完整优化器及模型状态，已完成训练自动跳过。环境/代码/输入发生变化会拒绝混用。


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/1013实验成果包与结论/10月13日_任务交付')
IMAGES = Path('/content/clip_jev_images')
WORK = Path('/content/drive/MyDrive/CLIP_Jev_CE_LS_学生后校准_20261023')
assert SOURCE.is_dir(), '请修改 SOURCE 为已解压的成果包目录。'
assert not WORK.resolve().is_relative_to(SOURCE.resolve()) and not SOURCE.resolve().is_relative_to(WORK.resolve()), '输入输出目录须相互独立。'
WORK.mkdir(parents=True,exist_ok=True)
ALLOW_HISTORICAL_TEST = False  # 完成训练/校准锁定后，如需历史复评再打开


## 3．写出内嵌代码

本 notebook 自带缓存核查和实验代码，无需另外上传 .py。


In [ ]:
import hashlib, importlib
SOURCES = {'cache_audit.py': '"""Read-only historical cache audit and disposable, unified training benchmark."""\nfrom pathlib import Path\nimport hashlib, json, time, math, random, platform, uuid, csv\nimport numpy as np\n\nREV = \'3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268\'\nMETHODS = [\'CE\', \'KD_raw\', \'KD_calibrated\', \'KD_base\', \'KD_hard\', \'CE_image\', \'CE_text\']\n\ndef check(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1048576), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef read(path): return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\ndef rows(path): return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8-sig\').splitlines() if s.strip()]\ndef save(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.partial\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef locate(source):\n    source = Path(source).resolve()\n    candidates = [source, source/\'reused_v1\']\n    candidates += list(source.glob(\'**/reused_v1\'))\n    matches = sorted(set(p for p in candidates if (p/\'students/training_lock.json\').is_file()))\n    check(len(matches) == 1, \'SOURCE 必须指向已解压的单个成果包或 Student_v1 目录；未唯一找到训练锁。\')\n    return matches[0]\n\ndef audit(source):\n    old = locate(source); lock = read(old/\'students/training_lock.json\')\n    files = {}; data = {}; seen = set(); seen_samples = set()\n    def record(p):\n        digest=sha(p); files[p.relative_to(old).as_posix()] = digest; return digest\n    revision = read(old/\'features/clip_revision.json\')\n    check(revision[\'revision\'] == REV, \'CLIP revision 不符\')\n    record(old/\'features/clip_revision.json\'); record(old/\'students/training_lock.json\')\n    # Only read held-out manifests to verify isolation; never read their features/predictions.\n    for split, n in [(\'train\',6000),(\'val_fit\',400),(\'val_select\',400),(\'test\',1000)]:\n        manifest = old/f\'handoff/manifests/{split}.jsonl\'; rr = rows(manifest); record(manifest)\n        ids = [str(r[\'image_id\']) for r in rr]; sid = [r[\'sample_id\'] for r in rr]\n        check(len(rr)==n and len(set(ids))==n and len(set(sid))==n, f\'{split} 数量/重复 ID 异常\')\n        check(not seen.intersection(ids) and not seen_samples.intersection(sid), \'划分间 ID 泄漏\')\n        seen.update(ids); seen_samples.update(sid)\n        check(all(r[\'label\'] in [0,1,2] and r[\'split\']==split for r in rr), \'标签或 split 异常\')\n        if split not in [\'train\',\'val_select\']: continue\n        p = old/f\'features/{split}.npz\'; meta_path = old/f\'features/{split}_meta.json\'\n        meta = read(meta_path); record(meta_path)\n        digest = record(p)\n        check(digest == meta[\'npz_sha256\'] == lock[\'feature_hashes\'][split], f\'{split} 特征哈希不符\')\n        check(meta[\'manifest_sha256\']==sha(manifest) and meta[\'revision\']==REV, \'特征来源不符\')\n        check(meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'] and meta[\'frozen_encoder\'], \'特征定义不符\')\n        with np.load(p, allow_pickle=False) as z: d = {k:z[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        x = d[\'x\']; v,t = x[:,:512],x[:,512:1024]\n        check(x.shape==(n,2048) and x.dtype==np.float32 and np.isfinite(x).all(), \'特征形状/精度/有限性异常\')\n        check(d[\'ids\'].tolist()==sid and d[\'image_ids\'].astype(str).tolist()==ids and d[\'y\'].tolist()==[r[\'label\'] for r in rr], \'缓存顺序或标签不符\')\n        check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4), \'特征未归一化\')\n        check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6), \'特征分段不符\')\n        image_audit = old/\'handoff/manifests/image_audit.jsonl\'\n        check(record(image_audit)==meta[\'image_audit_sha256\'], \'图片审计来源变化\')\n        d[\'rows\']=rr; data[split]=d\n    runpath = old/\'handoff/predictions/train/run.json\'; run = read(runpath); record(runpath)\n    check(run[\'saved_order\']==[\'E\',\'N\',\'C\'] and run[\'manifest_sha256\']==files[\'handoff/manifests/train.jsonl\'], \'Teacher 顺序/来源不符\')\n    logits = {}\n    image_rows = rows(old/\'handoff/manifests/image_audit.jsonl\')\n    image_map = {r[\'sample_id\']:r for r in image_rows}\n    check(len(image_map)==len(image_rows), \'图片审计重复 ID\')\n    for method in [\'jev\',\'base\']:\n        p = old/f\'handoff/predictions/train/{method}.jsonl\'; rr=rows(p); digest=record(p)\n        if method==\'jev\': check(digest==lock[\'teacher_train_sha256\'], \'Jev 缓存与训练锁不符\')\n        byid={r[\'sample_id\']:r for r in rr}\n        check(len(byid)==len(rr)==6000 and set(byid)==set(data[\'train\'][\'ids\']), \'教师缓存缺失/重复/多余样本\')\n        ordered=[byid[r[\'sample_id\']] for r in data[\'train\'][\'rows\']]\n        for r, pred in zip(data[\'train\'][\'rows\'],ordered):\n            check(pred[\'model\']==method and all(pred[k]==r[k] for k in [\'label\',\'image_id\',\'image_file\',\'hypothesis\',\'split\']), \'教师缓存内容错配\')\n            check(pred[\'image_sha256\']==image_map[r[\'sample_id\']][\'file_sha256\'], \'教师图片哈希错配\')\n        z=np.asarray([r[\'logits\'] for r in ordered],dtype=np.float64)\n        check(z.shape==(6000,3) and np.isfinite(z).all(), \'教师 logits 异常\'); logits[method]=z\n    calpath=old/\'handoff/results/calibration.json\'; cal=read(calpath)\n    check(record(calpath)==lock[\'calibration_sha256\'] and cal[\'labels\']==[\'E\',\'N\',\'C\'], \'校准文件不符\')\n    params=cal[\'models\'][\'jev\'][\'selected_params\']\n    check(np.isfinite([params[\'T\'],params[\'b\']]).all() and params[\'T\']>0, \'校准参数异常\')\n    package_manifest=old.parent/\'文件校验清单.json\'\n    if package_manifest.exists():\n        hashes=read(package_manifest)\n        for name,digest in files.items(): check(hashes[\'reused_v1/\'+name]==digest, \'成果包总清单不符：\'+name)\n    else:\n        raise ValueError(\'需要10月13日成果包的文件校验清单.json，以验证 base 缓存来源。请指定完整成果包。\')\n    report=dict(status=\'PASS\', source=str(old), input_sha256=files, clip_revision=REV,\n                counts={s:len(d[\'y\']) for s,d in data.items()}, label_order=[\'E\',\'N\',\'C\'],\n                scope=\'历史缓存完整性与来源；未重新读取原图或重新推理 CLIP/Teacher\', calibration=params)\n    return dict(old=old, data=data, logits=logits, params=params, report=report)\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\ndef make_head(method):\n    import torch.nn as nn\n    d,h=(512,2048) if method in [\'CE_image\',\'CE_text\'] else (2048,512)\n    return nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(.1),nn.Linear(h,3))\n\ndef select_features(x, method):\n    return x[:,:512] if method==\'CE_image\' else x[:,512:1024] if method==\'CE_text\' else x\n\ndef target(ctx, method, tau):\n    if method in [\'CE\',\'CE_image\',\'CE_text\']: return None\n    z=ctx[\'logits\'][\'base\' if method==\'KD_base\' else \'jev\'].copy()\n    if method==\'KD_hard\': return z.argmax(1)\n    if method==\'KD_calibrated\': z=(z+np.array([ctx[\'params\'][\'b\'],0,0]))/ctx[\'params\'][\'T\']\n    z=z/tau; z-=z.max(1,keepdims=True); p=np.exp(z); return (p/p.sum(1,keepdims=True)).astype(np.float32)\n\ndef loss_fn(z,y,q,method,alpha,tau):\n    import torch.nn.functional as F\n    ce=F.cross_entropy(z,y)\n    if q is None: return ce\n    kd=F.cross_entropy(z,q) if method==\'KD_hard\' else tau*tau*F.kl_div(F.log_softmax(z/tau,dim=-1),q,reduction=\'batchmean\')\n    return (1-alpha)*ce+alpha*kd\n\ndef benchmark(ctx, out, methods=METHODS, seed=20261010, alpha=.75, tau=2., batch=128, warmup=10, steps=50, repeats=3):\n    import torch\n    check(torch.cuda.is_available(), \'短跑要求 GPU；不自动用 CPU 替代计时。\')\n    check(0<=alpha<=1 and tau>0 and batch>0 and warmup>=1 and steps>=1 and repeats>=1, \'计时参数不合法\')\n    check(len(methods)>0 and len(set(methods))==len(methods) and set(methods)<=set(METHODS), \'未知或重复方法\')\n    out=Path(out); out.mkdir(parents=True, exist_ok=False)\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=True\n    env=dict(torch=str(torch.__version__), numpy=np.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),\n             python=platform.python_version(), cudnn=torch.backends.cudnn.version(), dtype=\'float32\', tf32_matmul=False, tf32_cudnn=True)\n    config=dict(methods=list(methods), seed=seed, alpha=alpha,tau=tau,batch=batch,warmup=warmup,steps=steps,repeats=repeats,\n                scope=\'GPU-resident cached features -> head training step; forward/loss/backward/AdamW; synchronized per step\',\n                excludes=[\'CLIP/Teacher inference\',\'disk read\',\'host-to-device copy\',\'validation\',\'checkpoint I/O\'], runtime_sha256=sha(__file__))\n    save(out/\'protocol.json\',dict(config=config,environment=env,cache_audit=ctx[\'report\']))\n    x=torch.tensor(ctx[\'data\'][\'train\'][\'x\'],device=\'cuda\'); y=torch.tensor(ctx[\'data\'][\'train\'][\'y\'],device=\'cuda\',dtype=torch.long)\n    check(batch<=len(y), \'batch 超过样本数\')\n    # Exact same sequence of full batches for every method and repeat; no tail batch in step timing.\n    g=torch.Generator(device=\'cpu\').manual_seed(seed)\n    batches=[]\n    while len(batches)<warmup+steps:\n        batches.extend(ix for ix in torch.randperm(len(y),generator=g).split(batch) if len(ix)==batch)\n    batches=[ix.to(\'cuda\') for ix in batches[:warmup+steps]]\n    raw=[]; summary=[]\n    for method in methods:\n        qn=target(ctx,method,tau); q=None if qn is None else torch.tensor(qn,device=\'cuda\',dtype=torch.long if method==\'KD_hard\' else torch.float32)\n        xx=select_features(x,method); times=[]; peaks=[]\n        for repeat in range(repeats):\n            seed_all(seed); model=make_head(method).cuda().train(); opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)\n            def step(ix):\n                opt.zero_grad(set_to_none=True)\n                loss=loss_fn(model(xx[ix]),y[ix],None if q is None else q[ix],method,alpha,tau)\n                loss.backward(); opt.step(); return loss\n            for ix in batches[:warmup]: step(ix)\n            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()\n            for j,ix in enumerate(batches[warmup:]):\n                torch.cuda.synchronize(); start=time.perf_counter(); loss=step(ix); torch.cuda.synchronize()\n                ms=(time.perf_counter()-start)*1000\n                value=float(loss.detach().cpu()); check(math.isfinite(value), \'短跑损失非有限\')\n                times.append(ms); raw.append(dict(method=method,repeat=repeat+1,step=j+1,ms=ms,loss=value))\n            peaks.append(torch.cuda.max_memory_allocated()/1024**2)\n            del opt,model; torch.cuda.empty_cache()\n        mean=float(np.mean(times)); median=float(np.median(times))\n        summary.append(dict(method=method,mean_step_ms=mean,median_step_ms=median,p95_step_ms=float(np.percentile(times,95)),\n                            samples_per_second=batch*1000/mean,peak_allocated_MiB=max(peaks),\n                            estimated_30_epochs_3_seeds_train_only_seconds=mean/1000*math.ceil(len(y)/batch)*30*3))\n        save(out/\'partial_results.json\',dict(summary=summary,raw=raw)); print(method, summary[-1],flush=True)\n        del q\n    save(out/\'results.json\',dict(summary=summary,raw=raw,note=\'工程短跑；全预算仅线性粗估，末批按整批估算；不含验证/保存，不能作为正式训练耗时或端到端加速比。\'))\n    with (out/\'timing_summary.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)\n    return summary\n', 'experiment.py': '"""Paired CE/label-smoothing/KD experiment and student temperature scaling."""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport csv, io, json, math, os, platform, time, uuid\nimport numpy as np\nimport cache_audit as base\n\ncheck, sha, read, rows, save = base.check, base.sha, base.read, base.rows, base.save\nDEFAULT = dict(seeds=[20261010,20261011,20261012], methods=[\'CE\',\'LS\',\'KD_raw\'],\n               label_smoothing=.1, alpha=.75, tau=2., epochs=30, batch=128,\n               lr=.001, weight_decay=.0001, temperature_min=.05, temperature_max=20., temperature_points=401)\n\ndef atomic_bytes(p, value):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_name(p.name+\'.\'+uuid.uuid4().hex+\'.partial\')\n    with tmp.open(\'xb\') as f: f.write(value); f.flush(); os.fsync(f.fileno())\n    tmp.replace(p)\n\ndef save_torch(p, value):\n    import torch\n    b=io.BytesIO(); torch.save(value,b); atomic_bytes(p,b.getvalue())\n\ndef freeze(p,value):\n    if Path(p).exists(): check(read(p)==value,\'锁定内容发生变化，请使用新 WORK：\'+str(p))\n    else: save(p,value)\n\n@contextmanager\ndef exclusive(work):\n    p=Path(work)/\'ACTIVE.lock\'\n    with p.open(\'x\',encoding=\'utf-8\') as f: json.dump(dict(pid=os.getpid(),note=\'确认旧进程结束后才能手动移除此锁\'),f)\n    try: yield\n    finally: p.unlink()\n\ndef environment(device=\'cuda\'):\n    import torch\n    check(device in [\'cuda\',\'cpu\'],\'device 不合法\')\n    if device==\'cuda\': check(torch.cuda.is_available(),\'请启用 Colab GPU\')\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\n    return dict(torch=str(torch.__version__),numpy=np.__version__,python=platform.python_version(),\n                device=device,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None,\n                cudnn=torch.backends.cudnn.version(),tf32=False,dtype=\'float32\')\n\ndef prepare(source,work,config=None,device=\'cuda\'):\n    config=json.loads(json.dumps(DEFAULT if config is None else config))\n    check(set(config)==set(DEFAULT),\'配置项不完整\')\n    check(config[\'methods\']==[\'CE\',\'LS\',\'KD_raw\'],\'本对照固定 CE/LS/KD_raw 三组\')\n    check(len(config[\'seeds\'])==3 and len(set(config[\'seeds\']))==3,\'需要三个不同种子\')\n    check(0<config[\'label_smoothing\']<1 and 0<=config[\'alpha\']<=1 and config[\'tau\']>0,\'损失参数异常\')\n    check(config[\'epochs\']>=1 and config[\'batch\']>=1 and config[\'lr\']>0 and config[\'weight_decay\']>=0,\'训练参数异常\')\n    check(0<config[\'temperature_min\']<1<config[\'temperature_max\'] and config[\'temperature_points\']>=3,\'温度网格异常\')\n    source,work=Path(source).resolve(),Path(work).resolve()\n    check(not work.is_relative_to(source) and not source.is_relative_to(work),\'SOURCE/WORK 必须相互独立\')\n    ctx=base.audit(source); work.mkdir(parents=True,exist_ok=True)\n    ctx.update(work=work,config=config,device=device)\n    protocol=dict(kind=\'CE_LS_student_temperature_v1\',config=config,environment=environment(device),\n                  cache=ctx[\'report\'],code={n:sha(Path(__file__).parent/n) for n in [\'experiment.py\',\'cache_audit.py\']},\n                  selection=\'val_select max macro-F1; tie min uncalibrated NLL; tie earliest epoch\',\n                  calibration=\'per checkpoint scalar positive temperature, val_fit NLL grid minimum; no bias\',\n                  evaluation=\'val_select descriptive reused validation; optional historical test already exposed\')\n    freeze(work/\'protocol.json\',protocol); ctx[\'protocol_sha256\']=sha(work/\'protocol.json\')\n    return ctx\n\ndef softmax(z):\n    z=np.asarray(z,dtype=np.float64); z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)\n\ndef nll(z,y):\n    z=np.asarray(z,dtype=np.float64); m=z.max(1)\n    return float(np.mean(m+np.log(np.exp(z-m[:,None]).sum(1))-z[np.arange(len(y)),y]))\n\ndef metrics(z,y):\n    z=np.asarray(z,dtype=np.float64); y=np.asarray(y,dtype=np.int64)\n    check(z.shape==(len(y),3) and len(y)>0 and np.isfinite(z).all(),\'预测异常\')\n    check(np.isin(y,[0,1,2]).all(),\'标签异常\')\n    p=softmax(z); pred=z.argmax(1); cm=np.zeros((3,3),dtype=np.int64); np.add.at(cm,(y,pred),1)\n    f=2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1)\n    confidence=p.max(1); correct=pred==y; bins=np.minimum((confidence*10).astype(int),9)\n    ece=sum(float((bins==b).mean())*abs(float(correct[bins==b].mean())-float(confidence[bins==b].mean())) for b in range(10) if np.any(bins==b))\n    return dict(accuracy=float(correct.mean()),macro_f1=float(f.mean()),nll=nll(z,y),\n                brier=float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1))),ece10=ece,\n                f1_E=float(f[0]),f1_N=float(f[1]),f1_C=float(f[2]))\n\ndef fit_temperature(z,y,cfg):\n    grid=np.unique(np.r_[1.,np.geomspace(cfg[\'temperature_min\'],cfg[\'temperature_max\'],cfg[\'temperature_points\'])])\n    values=[nll(z/t,y) for t in grid]\n    i=min(range(len(grid)),key=lambda i:(values[i],abs(math.log(grid[i]))))\n    return dict(T=float(grid[i]),fit_nll_raw=nll(z,y),fit_nll_calibrated=values[i],\n                at_boundary=i in [0,len(grid)-1],grid=grid.tolist(),nll_grid=values)\n\ndef checked_feature(ctx,split,folder):\n    folder=Path(folder); p=folder/f\'{split}.npz\'; meta=read(folder/f\'{split}_meta.json\')\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    check(meta[\'npz_sha256\']==sha(p) and meta[\'manifest_sha256\']==sha(manifest),\'新增特征哈希不符\')\n    check(meta[\'revision\']==base.REV and meta[\'frozen_encoder\'] and meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],\'新增特征定义不符\')\n    check(meta[\'image_audit_sha256\']==sha(ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'),\'图像审计来源变化\')\n    with np.load(p,allow_pickle=False) as a: d={k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n    x=d[\'x\']; check(x.shape==(len(rr),2048) and x.dtype==np.float32 and np.isfinite(x).all(),\'新增特征异常\')\n    check(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rr] and d[\'y\'].tolist()==[r[\'label\'] for r in rr] and d[\'image_ids\'].astype(str).tolist()==[str(r[\'image_id\']) for r in rr],\'新增特征 ID/标签错配\')\n    v,t=x[:,:512],x[:,512:1024]\n    check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4),\'新增特征未归一化\')\n    check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6),\'新增特征分段不符\')\n    d[\'rows\']=rr; return d\n\ndef feature_data(ctx,split):\n    if split in ctx[\'data\']: return ctx[\'data\'][split]\n    return checked_feature(ctx,split,ctx[\'work\']/\'features\')\n\ndef extract_missing(ctx,images,split=\'val_fit\'):\n    """Only new output cache; reuse requires verified manifest, payload and provenance."""\n    import torch\n    import torch.nn.functional as F\n    import importlib.metadata as md\n    from PIL import Image\n    from transformers import CLIPModel,CLIPProcessor\n    check(split in [\'val_fit\',\'test\'],\'只补齐 val_fit 或 test\')\n    if split==\'test\': check((ctx[\'work\']/\'student_calibration.json\').exists(),\'先锁定校准再准备测试输入\')\n    folder=ctx[\'work\']/\'features\'; folder.mkdir(exist_ok=True)\n    p=folder/f\'{split}.npz\'; mp=folder/f\'{split}_meta.json\'\n    if p.exists() and mp.exists(): return checked_feature(ctx,split,folder)\n    check(ctx[\'device\']==\'cuda\',\'正式特征提取需要 GPU\')\n    check(md.version(\'transformers\')==\'4.57.1\',\'请使用 transformers 4.57.1\')\n    images=Path(images).resolve(); audit_path=ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'\n    audit={r[\'sample_id\']:r for r in rows(audit_path)}\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    known=ctx[\'data\'][\'val_select\'][\'rows\'][:32]\n    def image_path(r):\n        path=(images/r[\'image_file\']).resolve()\n        check(path.is_relative_to(images),\'图片路径越界\')\n        check(path.is_file() and sha(path)==audit[r[\'sample_id\']][\'file_sha256\'],\'图片缺失/哈希不符：\'+str(path))\n        return path\n    for r in rr+known: image_path(r)\n    with exclusive(ctx[\'work\']):\n        model=CLIPModel.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV).to(\'cuda\',dtype=torch.float32).eval()\n        processor=CLIPProcessor.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV)\n        def encode(batch):\n            ims=[]\n            for r in batch:\n                with Image.open(image_path(r)) as im: ims.append(im.convert(\'RGB\'))\n            inp=processor(text=[r[\'hypothesis\'] for r in batch],images=ims,padding=True,truncation=True,max_length=77,return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():\n                output=model(**inp); v=F.normalize(output.image_embeds.float(),dim=-1); t=F.normalize(output.text_embeds.float(),dim=-1)\n                return torch.cat([v,t,abs(v-t),v*t],dim=-1).cpu().numpy()\n        try:\n            delta=float(abs(encode(known)-ctx[\'data\'][\'val_select\'][\'x\'][:32]).max())\n            check(delta<=1e-4,f\'与旧特征兼容检查失败：max abs={delta}；不要放宽阈值后混用。\')\n            provenance=dict(environment=environment(ctx[\'device\']),transformers=md.version(\'transformers\'),revision=base.REV)\n            freeze(folder/\'extraction_environment.json\',provenance)\n            values=[]; truncated=[]\n            for start in range(0,len(rr),32):\n                batch=rr[start:start+32]; values.append(encode(batch))\n                truncated += [r[\'sample_id\'] for r in batch if len(processor.tokenizer(r[\'hypothesis\'])[\'input_ids\'])>77]\n            b=io.BytesIO(); np.savez_compressed(b,x=np.concatenate(values),y=[r[\'label\'] for r in rr],ids=[r[\'sample_id\'] for r in rr],image_ids=[r[\'image_id\'] for r in rr])\n            atomic_bytes(p,b.getvalue()); save(mp,dict(npz_sha256=sha(p),manifest_sha256=sha(manifest),revision=base.REV,\n                 frozen_encoder=True,feature_order=[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],image_audit_sha256=sha(audit_path),\n                 compatibility_max_abs=delta,truncated_ids=truncated,extraction_environment=provenance))\n        finally:\n            del model; torch.cuda.empty_cache()\n    return checked_feature(ctx,split,folder)\n\ndef training_loss(z,y,q,method,cfg):\n    import torch.nn.functional as F\n    if method==\'LS\': return F.cross_entropy(z,y,label_smoothing=cfg[\'label_smoothing\'])\n    ce=F.cross_entropy(z,y)\n    if method==\'CE\': return ce\n    check(method==\'KD_raw\',\'未知方法\')\n    return (1-cfg[\'alpha\'])*ce+cfg[\'alpha\']*cfg[\'tau\']**2*F.kl_div(F.log_softmax(z/cfg[\'tau\'],dim=-1),q,reduction=\'batchmean\')\n\ndef predict(model,x,device):\n    import torch\n    model.eval(); zs=[]\n    with torch.inference_mode():\n        for start in range(0,len(x),256): zs.append(model(torch.as_tensor(x[start:start+256],device=device)).cpu().numpy())\n    return np.concatenate(zs)\n\ndef train_one(ctx,method,seed,stop_after_epoch=None):\n    """Epoch-atomic resume. Epoch-specific RNG makes resumed dropout/shuffle identical."""\n    import torch\n    cfg=ctx[\'config\']; device=ctx[\'device\']; folder=ctx[\'work\']/f\'runs/{method}_{seed}\'; folder.mkdir(parents=True,exist_ok=True)\n    statepath=folder/\'state.pt\'; donepath=folder/\'done.json\'\n    identity=dict(protocol_sha256=ctx[\'protocol_sha256\'],method=method,seed=seed)\n    if donepath.exists():\n        done=read(donepath); check(done[\'identity\']==identity and done[\'checkpoint_sha256\']==sha(folder/\'best.pt\'),\'已完成 checkpoint 变化\')\n        return done\n    check(not (ctx[\'work\']/\'selection.json\').exists(),\'模型已锁定，不能重新训练\')\n    a=ctx[\'data\'][\'train\']; b=ctx[\'data\'][\'val_select\']; base.seed_all(seed)\n    model=base.make_head(\'CE\').to(device); opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    x=torch.as_tensor(a[\'x\'],device=device); y=torch.as_tensor(a[\'y\'],dtype=torch.long,device=device)\n    q=torch.tensor(softmax(ctx[\'logits\'][\'jev\']/cfg[\'tau\']),dtype=torch.float32,device=device) if method==\'KD_raw\' else None\n    history=[]; best=(-float(\'inf\'),-float(\'inf\')); best_state=None; best_epoch=None; start=1\n    if statepath.exists():\n        s=torch.load(statepath,map_location=\'cpu\',weights_only=True); check(s[\'identity\']==identity,\'断点来源不一致\')\n        model.load_state_dict(s[\'model\']); opt.load_state_dict(s[\'optimizer\']); history=s[\'history\']; best=tuple(s[\'best\'])\n        best_state=s[\'best_state\']; best_epoch=s[\'best_epoch\']; start=s[\'epoch\']+1\n    for epoch in range(start,cfg[\'epochs\']+1):\n        base.seed_all(seed+100000*epoch); model.train(); total=0.; begun=time.perf_counter()\n        g=torch.Generator(device=\'cpu\').manual_seed(seed+epoch)\n        for ix in torch.randperm(len(y),generator=g).split(cfg[\'batch\']):\n            ix=ix.to(device); opt.zero_grad(set_to_none=True)\n            loss=training_loss(model(x[ix]),y[ix],None if q is None else q[ix],method,cfg)\n            check(bool(torch.isfinite(loss).item()),\'训练损失非有限\')\n            loss.backward(); opt.step(); total+=float(loss.detach().cpu())*len(ix)\n        z=predict(model,b[\'x\'],device); m=metrics(z,b[\'y\']); key=(m[\'macro_f1\'],-m[\'nll\'])\n        if key>best:\n            best=key; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        history.append(dict(epoch=epoch,loss=total/len(y),validation=m,train_and_validation_seconds=time.perf_counter()-begun))\n        save_torch(statepath,dict(identity=identity,epoch=epoch,model=model.state_dict(),optimizer=opt.state_dict(),history=history,\n                                  best=list(best),best_state=best_state,best_epoch=best_epoch))\n        print(method,seed,f\'{epoch}/{cfg["epochs"]}\',f\'val F1={m["macro_f1"]:.4f}\',flush=True)\n        if stop_after_epoch is not None and epoch>=stop_after_epoch and epoch<cfg[\'epochs\']: return None\n    check(best_state is not None and len(history)==cfg[\'epochs\'],\'训练不完整\')\n    save_torch(folder/\'best.pt\',best_state); save(folder/\'history.json\',history)\n    done=dict(identity=identity,method=method,seed=seed,best_epoch=best_epoch,validation_macro_f1=best[0],validation_nll=-best[1],\n              checkpoint=str((folder/\'best.pt\').relative_to(ctx[\'work\'])),checkpoint_sha256=sha(folder/\'best.pt\'),history_sha256=sha(folder/\'history.json\'))\n    save(donepath,done); return done\n\ndef train_all(ctx):\n    environment(ctx[\'device\']); selected=[]\n    with exclusive(ctx[\'work\']):\n        for seed in ctx[\'config\'][\'seeds\']:\n            for method in ctx[\'config\'][\'methods\']: selected.append(train_one(ctx,method,seed))\n        freeze(ctx[\'work\']/\'selection.json\',dict(protocol_sha256=ctx[\'protocol_sha256\'],models=selected))\n    return selected\n\ndef selected_models(ctx):\n    selection=read(ctx[\'work\']/\'selection.json\'); check(selection[\'protocol_sha256\']==ctx[\'protocol_sha256\'],\'选模协议变化\')\n    expected={(m,s) for m in ctx[\'config\'][\'methods\'] for s in ctx[\'config\'][\'seeds\']}\n    check(len(selection[\'models\'])==len(expected) and {(m[\'method\'],m[\'seed\']) for m in selection[\'models\']}==expected,\'选模不完整\')\n    for m in selection[\'models\']: check(sha(ctx[\'work\']/m[\'checkpoint\'])==m[\'checkpoint_sha256\'],\'checkpoint 损坏\')\n    return selection[\'models\']\n\ndef model_logits(ctx,model_info,data):\n    import torch\n    model=base.make_head(\'CE\').to(ctx[\'device\'])\n    model.load_state_dict(torch.load(ctx[\'work\']/model_info[\'checkpoint\'],map_location=\'cpu\',weights_only=True))\n    return predict(model,data[\'x\'],ctx[\'device\'])\n\ndef calibrate(ctx):\n    data=feature_data(ctx,\'val_fit\'); models=selected_models(ctx); path=ctx[\'work\']/\'student_calibration.json\'\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),\n                val_fit_npz_sha256=sha(ctx[\'work\']/\'features/val_fit.npz\'),val_fit_meta_sha256=sha(ctx[\'work\']/\'features/val_fit_meta.json\'))\n    if path.exists():\n        result=read(path); check(result[\'inputs\']==inputs,\'校准输入变化\'); return result\n    with exclusive(ctx[\'work\']):\n        result=[]\n        for m in models:\n            z=model_logits(ctx,m,data); fit=fit_temperature(z,data[\'y\'],ctx[\'config\'])\n            check(np.array_equal(z.argmax(1),(z/fit[\'T\']).argmax(1)),\'正温度不应改变 argmax\')\n            result.append(dict(method=m[\'method\'],seed=m[\'seed\'],checkpoint_sha256=m[\'checkpoint_sha256\'],**fit))\n            print(\'temperature\',m[\'method\'],m[\'seed\'],fit[\'T\'],\'boundary=\',fit[\'at_boundary\'],flush=True)\n        value=dict(inputs=inputs,models=result,scope=\'fit NLL on val_fit; this fit loss is not evaluation performance\')\n        freeze(path,value)\n    return value\n\ndef csvwrite(p,records):\n    if not records: return\n    b=io.StringIO(newline=\'\'); w=csv.DictWriter(b,fieldnames=list(records[0])); w.writeheader(); w.writerows(records)\n    atomic_bytes(p,b.getvalue().encode(\'utf-8-sig\'))\n\ndef report(ctx,split=\'val_select\',allow_historical_test=False):\n    check(split in [\'val_select\',\'test\'],\'报告划分不合法\')\n    check(split!=\'test\' or allow_historical_test,\'历史测试复评开关未开启\')\n    models=selected_models(ctx); calibration=calibrate(ctx); data=feature_data(ctx,split)\n    out=ctx[\'work\']/f\'reports/{split}\'; out.mkdir(parents=True,exist_ok=True)\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),calibration_sha256=sha(ctx[\'work\']/\'student_calibration.json\'),\n                feature_sha256=sha((ctx[\'old\'] if split==\'val_select\' else ctx[\'work\'])/f\'features/{split}.npz\'))\n    if (out/\'COMPLETE.json\').exists():\n        done=read(out/\'COMPLETE.json\'); check(done[\'inputs\']==inputs,\'已完成报告输入改变\')\n        for name,h in done[\'files\'].items(): check(sha(out/name)==h,\'报告文件损坏\')\n        return out\n    with exclusive(ctx[\'work\']):\n        values=[]; predictions=[]\n        for m in models:\n            fit=next(c for c in calibration[\'models\'] if c[\'method\']==m[\'method\'] and c[\'seed\']==m[\'seed\'])\n            check(fit[\'checkpoint_sha256\']==m[\'checkpoint_sha256\'],\'校准与模型错配\')\n            z=model_logits(ctx,m,data)\n            for variant,T in [(\'raw\',1.),(\'temperature\',fit[\'T\'])]:\n                zz=z.astype(np.float64)/T; p=softmax(zz)\n                values.append(dict(method=m[\'method\'],seed=m[\'seed\'],variant=variant,T=T,**metrics(zz,data[\'y\'])))\n                for i,sid in enumerate(data[\'ids\']):\n                    predictions.append(dict(sample_id=str(sid),image_id=str(data[\'image_ids\'][i]),label=int(data[\'y\'][i]),method=m[\'method\'],seed=m[\'seed\'],variant=variant,\n                                            logit_E=float(zz[i,0]),logit_N=float(zz[i,1]),logit_C=float(zz[i,2]),p_E=float(p[i,0]),p_N=float(p[i,1]),p_C=float(p[i,2]),prediction=int(zz[i].argmax())))\n            a,b=values[-2:]; check(a[\'accuracy\']==b[\'accuracy\'] and a[\'macro_f1\']==b[\'macro_f1\'],\'温度校准意外改变分类\')\n        metric_names=[\'accuracy\',\'macro_f1\',\'nll\',\'brier\',\'ece10\',\'f1_E\',\'f1_N\',\'f1_C\']; summary=[]; deltas=[]\n        for method in ctx[\'config\'][\'methods\']:\n            for variant in [\'raw\',\'temperature\']:\n                group=[r for r in values if r[\'method\']==method and r[\'variant\']==variant]\n                summary.append(dict(method=method,variant=variant,**{k+suffix:float(fn([r[k] for r in group])) for k in metric_names for suffix,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda a:np.std(a,ddof=1))]}))\n        def get(method,seed,variant): return next(r for r in values if r[\'method\']==method and r[\'seed\']==seed and r[\'variant\']==variant)\n        for seed in ctx[\'config\'][\'seeds\']:\n            for variant in [\'raw\',\'temperature\']:\n                for a,b in [(\'LS\',\'CE\'),(\'KD_raw\',\'CE\'),(\'KD_raw\',\'LS\')]:\n                    ra,rb=get(a,seed,variant),get(b,seed,variant)\n                    deltas.append(dict(comparison=a+\' minus \'+b,seed=seed,variant=variant,**{k:ra[k]-rb[k] for k in metric_names}))\n            for method in ctx[\'config\'][\'methods\']:\n                ra,rb=get(method,seed,\'temperature\'),get(method,seed,\'raw\')\n                deltas.append(dict(comparison=method+\' temperature minus raw\',seed=seed,variant=\'postcalibration\',**{k:ra[k]-rb[k] for k in metric_names}))\n        csvwrite(out/\'per_seed.csv\',values); csvwrite(out/\'summary.csv\',summary); csvwrite(out/\'paired_deltas.csv\',deltas); csvwrite(out/\'predictions.csv\',predictions)\n        save(out/\'metrics.json\',dict(inputs=inputs,per_seed=values,summary=summary,paired_deltas=deltas))\n        note=(\'验证选择集，已用于 checkpoint 选择，仅作开发性描述。\' if split==\'val_select\' else \'历史测试集已被查看；本轮复评不能称为全新独立确认。\')\n        text=\'# CE / 标签平滑 / 学生后校准\\n\\n\'+note+\'\\n\\n所有准确率、F1、ECE 用 0–1 原值；差值也是原值。seed_sd 为三个种子之间的样本标准差，不是置信区间。NLL/Brier/ECE 越低越好。\\n\\n温度只改变概率，不改变分类；不保证评价集的概率指标一定改善。边界温度见 student_calibration.json，不据测试结果扩网格。\\n\\n|方法|版本|Macro-F1 均值|NLL 均值|Brier 均值|ECE 均值|\\n|---|---|---:|---:|---:|---:|\\n\'\n        for r in summary: text+=f"|{r[\'method\']}|{r[\'variant\']}|{r[\'macro_f1_mean\']:.4f}|{r[\'nll_mean\']:.4f}|{r[\'brier_mean\']:.4f}|{r[\'ece10_mean\']:.4f}|\\n"\n        atomic_bytes(out/\'README.md\',text.encode(\'utf-8\'))\n        names=[\'per_seed.csv\',\'summary.csv\',\'paired_deltas.csv\',\'predictions.csv\',\'metrics.json\',\'README.md\']\n        save(out/\'COMPLETE.json\',dict(inputs=inputs,files={n:sha(out/n) for n in names}))\n    return out\n'}
DIGESTS = {'cache_audit.py': 'fc464669c08fd73dfd0449a92ba3b37e28595debc64aa32f0b11eac559cebd65', 'experiment.py': '55a0631475ffd30481945da0c0c7a6923312aaee2cbd6d5b62bf0dfb42fccad7'}
CODE = WORK/'code'
CODE.mkdir(exist_ok=True)
for name,source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==DIGESTS[name]
    dest=CODE/name
    if dest.exists():
        assert dest.read_text(encoding='utf-8')==source, '代码版本改变，请保留旧 WORK 并另建新实验目录。'
    else:
        dest.write_text(source,encoding='utf-8')
sys.path.insert(0,str(CODE))
import cache_audit, experiment
importlib.reload(cache_audit); importlib.reload(experiment)
assert Path(experiment.__file__).resolve()==(CODE/'experiment.py').resolve()
print('运行代码已就绪。')


## 4．核查缓存、锁定配置

核查 train=6000、val_select=400，四个划分的图像隔离，CLIP 特征及 Teacher 缓存哈希/ID/标签/图片来源。此阶段不读取测试预测、不评分。缓存 PASS 不等于重新推理了历史模型，也不能证明包外未使用过这些样本。

val_fit 的 400 条用于学生温度拟合；val_select 用于 checkpoint 选择及开发性对照。它们都是已使用的验证数据，不能充当新独立确认集。


In [ ]:
CONFIG = dict(experiment.DEFAULT)
# 如要改变预设，必须在首次运行本格前决定；之后新建 WORK。
ctx = experiment.prepare(SOURCE,WORK,config=CONFIG,device='cuda')
print('缓存 PASS；本轮配置：')
print(CONFIG)


## 5．补齐 val_fit 的冻结 CLIP 特征

原成果包缺少 val_fit 特征，本格使用固定 CLIP revision 提取并缓存。先重提取 32 条已有 val_select 特征，检查最大绝对差 ≤1e-4；随后检查原图哈希、提取 400 条 val_fit。若原图缺失或兼容检查失败，会中止，不会放宽标准混用。

完整缓存已存在时仅核查并复用。特征提取中断可重跑本格，只重建新 WORK 中未完成的文件。这里不拟合温度、不查看 val_fit 成绩。


In [ ]:
fit_features=experiment.extract_missing(ctx,IMAGES,'val_fit')
print('val_fit 特征就绪：',fit_features['x'].shape)
del fit_features


## 6．训练 CE / LS / Jev raw KD（三种子）

只使用 train 更新模型；每轮在 val_select 选模，val_fit 不参与梯度或 checkpoint 选择。所有方法均重新训练，避免跨环境直接比较新 LS 与历史 CE。

每轮断点原子保存；恢复时回到最近完成轮。每轮设置固定随机种子和独立批次生成器，使续跑与连续运行的随机序列一致。同一 WORK 只允许一个会话写入；若异常退出留下 ACTIVE.lock，仅在确认旧会话已停止后手动删除该锁。


In [ ]:
models=experiment.train_all(ctx)
display(pd.DataFrame([{k:m[k] for k in ['method','seed','best_epoch','validation_macro_f1','validation_nll']} for m in models]))
print('9 个模型已锁定：',WORK/'selection.json')


## 7．学生后校准：每个模型拟合一个正温度 T

先冻结选定 checkpoint，然后在 val_fit 上最小化学生 NLL。网格预先固定为 [0.05,20] 的 401 个对数点，额外包含 T=1；这是网格内最优，不宣称连续最优。校准形式为 `softmax(student_logits/T)`，不加类别偏置、不更新学生权重。

将报告温度及是否触及边界。触边保留记录，不根据后续测试分数扩范围。拟合集 NLL 只是拟合诊断，不是效果评价；后校准也不保证验证/测试 NLL、Brier 或 ECE 全都改善。


In [ ]:
calibration=experiment.calibrate(ctx)
display(pd.DataFrame([{k:r[k] for k in ['method','seed','T','fit_nll_raw','fit_nll_calibrated','at_boundary']} for r in calibration['models']]))
print('学生校准参数已锁定：',WORK/'student_calibration.json')


## 8．生成开发阶段验证对照

输出三种方法 × 三个种子 × raw/temperature。包括 Accuracy、Macro-F1、各类 F1、NLL、三类平方误差之和的 Brier、10 等宽置信区间的 ECE。

表中均为 0–1 原值（NLL 为自然对数）；F1/Accuracy/ECE 的差值乘 100 才是百分点。NLL/Brier/ECE 越低越好。报告三种子均值与样本标准差；标准差不是置信区间。逐种子配对差包括 LS−CE、KD−CE、KD−LS、各学生校准后−校准前。

val_select 已用于选 checkpoint，所以这里的结果仅供开发性描述。温度校准后的分类指标应保持一致，代码会检查。


In [ ]:
VALIDATION_REPORT=experiment.report(ctx,'val_select')
display(pd.read_csv(VALIDATION_REPORT/'summary.csv'))
display(pd.read_csv(VALIDATION_REPORT/'paired_deltas.csv'))
print('报告目录：',VALIDATION_REPORT)


## 9．可选：历史测试集复评（默认关闭）

只有模型和温度已锁定后才能运行。本格补齐 1000 条历史 test 特征，应用固定 checkpoint/T，绝不再选 ε、epoch 或 T。此 test 曾经被查看，结果属于历史复评，不能支持“全新独立确认”的表述。

本轮没有新的独立确认数据。若之后根据这些结果修改方法，需要另设未用于决策的评价方案。


In [ ]:
if ALLOW_HISTORICAL_TEST:
    experiment.extract_missing(ctx,IMAGES,'test')
    TEST_REPORT=experiment.report(ctx,'test',allow_historical_test=True)
    display(pd.read_csv(TEST_REPORT/'summary.csv'))
    print('历史测试复评：',TEST_REPORT)
else:
    print('已跳过历史测试；CE/LS 开发性比较与学生后校准已经完成。')


## 10．导出成果包

包含协议、校准网格与参数、九个 best checkpoint、每轮历史、逐样本预测、汇总与配对差、代码和特征来源记录。不打包原始图片、重复特征或体积较大的优化器断点；断点仍在 WORK 中。

读表重点：① LS 相对 CE 是否改善；② KD 相对 LS 是否仍有收益；③学生后校准是否降低 NLL/Brier/ECE，且分类指标不变。保留全部种子及失败/无提升结果，不只挑最好种子。


In [ ]:
import zipfile
from datetime import datetime, timezone, timedelta
import uuid
assert (WORK/'reports/val_select/COMPLETE.json').is_file(), '请先完成验证报告。'
files=[WORK/'protocol.json',WORK/'selection.json',WORK/'student_calibration.json']
files+=list((WORK/'code').glob('*.py'))
files+=list((WORK/'features').glob('*_meta.json'))
files+=list((WORK/'features').glob('extraction_environment.json'))
for model in experiment.selected_models(ctx):
    folder=(WORK/model['checkpoint']).parent
    files += [folder/'best.pt',folder/'history.json',folder/'done.json']
files += [p for p in (WORK/'reports').rglob('*') if p.is_file()]
stamp=datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
ZIP=WORK/f'CE_LS_学生后校准_{stamp}.zip'
with zipfile.ZipFile(ZIP,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(files)): z.write(p,p.relative_to(WORK).as_posix())
print('完成。成果包已保存：',ZIP)
